# 01 - Auditoria inicial dos dados

Esta etapa verifica o arquivo, o esquema e a qualidade básica antes de qualquer modelagem. O objetivo é tornar explícito o que sabemos dos dados, não selecionar um modelo nem inferir significado clínico dos nomes codificados.

Coloque `final_treino_val.csv` em `data/raw/`. O arquivo é ignorado pelo Git. O regulamento define `desfecho=1` como classe positiva e `id` como identificador. Não use `id` como atributo preditor.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/raw/final_treino_val.csv")
TARGET = "desfecho"
ID_COLUMN = "id"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH.resolve()}. "
        "Copie o CSV local para data/raw/ sem adicioná-lo ao Git."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()
print(f"Dimensões: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## Esquema e integridade

Vamos confirmar a presença do identificador e do alvo, verificar os rótulos observados, valores ausentes, duplicatas e cardinalidade. Tipos inteiros, por si só, não significam que uma variável deva ser tratada como contínua; a classificação de tipos deve considerar o regulamento.

In [ ]:
required_columns = {ID_COLUMN, TARGET}
missing_required = required_columns - set(df.columns)
if missing_required:
    raise ValueError(f"Colunas obrigatórias ausentes: {sorted(missing_required)}")

print("Tipos de dados:")
display(df.dtypes.rename("dtype").to_frame())

quality = pd.Series(
    {
        "linhas": len(df),
        "colunas": df.shape[1],
        "linhas duplicadas": int(df.duplicated().sum()),
        "IDs ausentes": int(df[ID_COLUMN].isna().sum()),
        "IDs duplicados": int(df[ID_COLUMN].duplicated().sum()),
        "alvos ausentes": int(df[TARGET].isna().sum()),
    },
)
display(quality.rename("valor").to_frame())
print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).rename("ausentes").to_frame())

observed_target = set(df[TARGET].dropna().unique())
if not observed_target.issubset({0, 1}):
    raise ValueError(f"Rótulos inesperados em {TARGET}: {observed_target}")
print("Rótulos observados:", sorted(observed_target))

In [ ]:
feature_columns = [c for c in df.columns if c not in {ID_COLUMN, TARGET}]
cardinality = df[feature_columns].nunique(dropna=False).sort_values()
display(cardinality.rename("valores distintos").to_frame())

target_counts = df[TARGET].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame(
    {"contagem": target_counts, "proporcao": target_counts / len(df)}
)
display(target_summary)

ax = sns.countplot(data=df, x=TARGET, order=[0, 1], color="steelblue")
ax.set(title="Distribuição do desfecho", xlabel="Desfecho (0 = cura, 1 = óbito)", ylabel="Número de registros")
plt.tight_layout()
plt.show()

## Perfil das variáveis

A tabela abaixo registra a cardinalidade e estatísticas descritivas sem atribuir significado clínico aos códigos. O regulamento identifica diversas variáveis como binárias, uma sociodemográfica como discreta e outras como nominais; essas distinções serão importantes ao construir pipelines de pré-processamento.

Uma diferença observada entre grupos é descritiva e não demonstra causalidade. Nenhuma decisão de modelagem ou limiar deve ser tomada com base no conjunto de teste final.

In [ ]:
display(df[feature_columns].describe(include="all").T)

# Proporções do alvo por valor observado; útil para inspeção, sem interpretação causal.
for column in feature_columns:
    table = pd.crosstab(df[column], df[TARGET], normalize="index")
    table.columns = [f"proporcao_desfecho_{value}" for value in table.columns]
    print(f"{column} (n valores={df[column].nunique(dropna=False)})")
    display(table)

## Registro da auditoria e próximos passos

Ao terminar, registre no README ou no histórico do experimento quaisquer anomalias relevantes. A próxima etapa será definir uma estratégia de validação estratificada e uma métrica de comparação alinhada à AUC-ROC, antes de ajustar modelos. O identificador será mantido separado para rastreio/submissão, sem entrar em `X`.